# 03 – Linear Probe & Adapters (Phase 4)

Operates entirely on **cached features** — no GPU needed.

Pipeline:
1. Zero-shot baseline (reference from Phase 3)
2. Logistic regression on frozen features (grid search over C)
3. Prototype / nearest-class-mean classifier
4. Tip-Adapter cache model
5. WiSE-FT blend (zero-shot + adapted logits)
6. Best config → submission

**Phase 4 exit gate:** beats pure zero-shot on validation with stable margin.

In [ ]:
import sys, pathlib
sys.path.insert(0, "/kaggle/input/datasets/somyajitsaha/food-retrieval-src")

import numpy as np
import pandas as pd
import torch

from src.config import seed_everything
seed_everything(42)

import sklearn
print(f"numpy={np.__version__}  sklearn={sklearn.__version__}  torch={torch.__version__}")

DATA_DIR = pathlib.Path("/kaggle/input/competitions/match-the-dish-to-its-rizz")
OUT_DIR  = pathlib.Path("/kaggle/working")
(OUT_DIR / "submissions").mkdir(exist_ok=True)
print("DATA_DIR exists:", DATA_DIR.exists())

In [ ]:
# ── Load data & text matcher ──────────────────────────────────────────────────
from src.data import load_csvs, parse_val_ground_truth
from src.text_matching import build_surface_form_dict, TextMatcher

dfs          = load_csvs(DATA_DIR)
train_df     = dfs["train"]
val_gallery  = dfs["val_gallery"]
val_queries  = dfs["val_queries"]
val_gt_df    = dfs["val_gt"]
test_gallery = dfs["test_gallery"]
test_queries = dfs["test_queries"]
meta         = dfs["category_metadata"]
val_gt       = parse_val_ground_truth(val_gt_df)

cat_id_list = meta["category_id"].tolist()
cat_id2idx  = {c: i for i, c in enumerate(cat_id_list)}
K           = len(cat_id_list)

matcher = TextMatcher(build_surface_form_dict(meta), meta_df=meta)
print(f"K={K}")

In [ ]:
# ── Load cached features (SigLIP = best from Phase 3) ────────────────────────
from src.features import load_features, load_text_features

# Primary model
PRIMARY   = "siglip_so400m"
SECONDARY = "vitl14_openai"

feat_dir_p = OUT_DIR / "features" / PRIMARY
feat_dir_s = OUT_DIR / "features" / SECONDARY

X_train_p, train_ids = load_features(feat_dir_p, "train", gallery_df=train_df)
X_val_p,   val_ids   = load_features(feat_dir_p, "val",   gallery_df=val_gallery)
X_test_p,  test_ids  = load_features(feat_dir_p, "test",  gallery_df=test_gallery)
T_proto_p, _         = load_text_features(feat_dir_p, tag="text")

X_train_s, _ = load_features(feat_dir_s, "train", gallery_df=train_df)
X_val_s,   _ = load_features(feat_dir_s, "val",   gallery_df=val_gallery)
X_test_s,  _ = load_features(feat_dir_s, "test",  gallery_df=test_gallery)
T_proto_s, _ = load_text_features(feat_dir_s, tag="text")

# Training labels as zero-based indices
id2cat  = dict(zip(train_df["image_id"], train_df["category_id"]))
y_str   = [id2cat[i] for i in train_ids]
y_enc   = np.array([cat_id2idx[c] for c in y_str])

print(f"Train: {X_train_p.shape}  Val: {X_val_p.shape}  Test: {X_test_p.shape}")

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────
from src.metrics import evaluate, print_metrics
from src.ensemble import temperature_scale

def score_to_preds(S, query_df, matcher, cat_id2idx, gallery_ids, top=20):
    pred_dict = {}
    for _, row in query_df.iterrows():
        cat_id, _, _ = matcher.predict(str(row["description"]))
        cat_idx = cat_id2idx.get(cat_id, 0)
        order   = np.argsort(-S[:, cat_idx], kind="stable")
        pred_dict[str(row["query_id"])] = [gallery_ids[j] for j in order[:top]]
    return pred_dict

results = []   # accumulate all experiment rows

In [ ]:
# ── 1. Zero-shot reference (SigLIP) ──────────────────────────────────────────
S_zs_val  = temperature_scale(X_val_p  @ T_proto_p.T, T=1.0, to_prob=True)
S_zs_test = temperature_scale(X_test_p @ T_proto_p.T, T=1.0, to_prob=True)

zs_preds   = score_to_preds(S_zs_val, val_queries, matcher, cat_id2idx, val_ids)
zs_metrics = evaluate(zs_preds, val_gt)
results.append({"method": "zero-shot SigLIP", **zs_metrics})
print("Zero-shot SigLIP (reference):")
print_metrics(zs_metrics)

In [ ]:
# ── 2. Logistic regression – grid search over C ───────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Logistic regression C grid search (5-fold CV on train)...")
C_values  = [0.01, 0.1, 1.0, 10.0, 100.0]
cv_scores = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for C in C_values:
    clf    = LogisticRegression(C=C, max_iter=2000, solver="lbfgs",
                                 multi_class="multinomial", random_state=42, n_jobs=-1)
    scores = cross_val_score(clf, X_train_p, y_enc, cv=skf,
                              scoring="accuracy", n_jobs=-1)
    cv_scores.append((C, scores.mean(), scores.std()))
    print(f"  C={C:7.2f}  CV acc={scores.mean():.4f} ± {scores.std():.4f}")

best_C = max(cv_scores, key=lambda x: x[1])[0]
print(f"\nBest C: {best_C}")

In [ ]:
# Fit on full train with best C
clf_best = LogisticRegression(C=best_C, max_iter=2000, solver="lbfgs",
                               multi_class="multinomial", random_state=42, n_jobs=-1)
clf_best.fit(X_train_p, y_enc)
print(f"Train acc: {clf_best.score(X_train_p, y_enc):.4f}")

S_lr_val  = clf_best.predict_proba(X_val_p)
S_lr_test = clf_best.predict_proba(X_test_p)

lr_preds   = score_to_preds(S_lr_val, val_queries, matcher, cat_id2idx, val_ids)
lr_metrics = evaluate(lr_preds, val_gt)
results.append({"method": f"LogReg C={best_C}", **lr_metrics})
print(f"\nLogistic Regression (C={best_C}):")
print_metrics(lr_metrics)

In [ ]:
# ── 3. Prototype / nearest-class-mean ─────────────────────────────────────────
# Average train features per class → L2-normalise → cosine scoring
proto = np.zeros((K, X_train_p.shape[1]), dtype=np.float32)
counts = np.zeros(K, dtype=np.int32)
for feat, label in zip(X_train_p, y_enc):
    proto[label]  += feat
    counts[label] += 1
counts = np.maximum(counts, 1)
proto /= counts[:, None]
proto /= np.linalg.norm(proto, axis=1, keepdims=True) + 1e-9

S_proto_val  = temperature_scale(X_val_p  @ proto.T, T=1.0, to_prob=True)
proto_preds  = score_to_preds(S_proto_val, val_queries, matcher, cat_id2idx, val_ids)
proto_metrics= evaluate(proto_preds, val_gt)
results.append({"method": "Prototype (NCM)", **proto_metrics})
print("Prototype / NCM:")
print_metrics(proto_metrics)

In [ ]:
# ── 4. WiSE-FT blend: zero-shot + logistic regression ────────────────────────
from src.ensemble import blend_zeroshot_finetuned

# Raw logit versions for blending (before softmax)
S_zs_raw_val  = X_val_p  @ T_proto_p.T   # cosine scores
S_lr_raw_val  = clf_best.decision_function(X_val_p)  # log-odds

# Normalise each to zero mean / unit std per row for comparable scales
def row_norm(S):
    mu  = S.mean(axis=1, keepdims=True)
    std = S.std(axis=1, keepdims=True) + 1e-9
    return (S - mu) / std

S_zs_n = row_norm(S_zs_raw_val)
S_lr_n = row_norm(S_lr_raw_val)

best_alpha, best_rp = 0.0, -1.0
print("WiSE-FT alpha sweep:")
for alpha in np.arange(0.0, 1.05, 0.1):
    S_blend = temperature_scale(
        blend_zeroshot_finetuned(S_zs_n, S_lr_n, alpha=float(alpha)),
        T=1.0, to_prob=True)
    preds  = score_to_preds(S_blend, val_queries, matcher, cat_id2idx, val_ids)
    rp     = evaluate(preds, val_gt)["r_precision"]
    marker = " ←" if rp > best_rp else ""
    print(f"  alpha={alpha:.1f}  R-prec={rp:.4f}{marker}")
    if rp > best_rp:
        best_rp, best_alpha = rp, float(alpha)

print(f"\nBest alpha: {best_alpha:.1f}  R-prec={best_rp:.4f}")

In [ ]:
# Evaluate best WiSE-FT blend
S_zs_raw_test = X_test_p @ T_proto_p.T
S_lr_raw_test = clf_best.decision_function(X_test_p)

S_blend_val  = temperature_scale(
    blend_zeroshot_finetuned(row_norm(S_zs_raw_val),  row_norm(S_lr_raw_val),  alpha=best_alpha),
    T=1.0, to_prob=True)
S_blend_test = temperature_scale(
    blend_zeroshot_finetuned(row_norm(S_zs_raw_test), row_norm(S_lr_raw_test), alpha=best_alpha),
    T=1.0, to_prob=True)

blend_preds   = score_to_preds(S_blend_val, val_queries, matcher, cat_id2idx, val_ids)
blend_metrics = evaluate(blend_preds, val_gt)
results.append({"method": f"WiSE-FT α={best_alpha:.1f}", **blend_metrics})
print(f"WiSE-FT (α={best_alpha:.1f}):")
print_metrics(blend_metrics)

In [ ]:
# ── 5. Multi-model ensemble (SigLIP + ViT-L/14) ───────────────────────────────
from src.ensemble import LogitEnsemble

S_zs_val_s  = X_val_s  @ T_proto_s.T   # ViT-L/14 cosine

# Uniform ensemble of both zero-shot models
ens = LogitEnsemble()
ens.add(S_zs_raw_val, weight=1.0, temperature=1.0)    # SigLIP
ens.add(S_zs_val_s,   weight=1.0, temperature=1.0)    # ViT-L/14
S_ens_val = ens.combine()

ens_preds   = score_to_preds(S_ens_val, val_queries, matcher, cat_id2idx, val_ids)
ens_metrics = evaluate(ens_preds, val_gt)
results.append({"method": "Ensemble SigLIP+ViTL", **ens_metrics})
print("Ensemble (SigLIP + ViT-L/14, equal weights):")
print_metrics(ens_metrics)

In [ ]:
# ── 6. Summary table ──────────────────────────────────────────────────────────
print("\n" + "="*70)
print("PHASE 4 SUMMARY")
print("="*70)
summary = pd.DataFrame(results)
print(summary[["method","r_precision","map@20","p@5","hit@1","mrr@20"]].to_string(index=False))

# Pick best method by R-precision
best_row  = max(results, key=lambda x: x["r_precision"])
best_method = best_row["method"]
print(f"\n→ Best method: {best_method}  R-prec={best_row['r_precision']:.4f}")

In [ ]:
# ── 7. Build submission with best method ─────────────────────────────────────
from src.submit import build_submission, validate_submission
import datetime

# Use whichever score matrix won
if "Ensemble" in best_method:
    S_zs_test_s  = X_test_s @ T_proto_s.T
    ens_t = LogitEnsemble()
    ens_t.add(S_zs_raw_test, weight=1.0, temperature=1.0)
    ens_t.add(S_zs_test_s,   weight=1.0, temperature=1.0)
    S_final_test = ens_t.combine()
    tag = "ens_siglip_vitl14"
elif "WiSE" in best_method:
    S_final_test = S_blend_test
    tag = f"wiseft_a{best_alpha:.1f}_siglip"
elif "LogReg" in best_method:
    S_final_test = S_lr_test
    tag = f"logreg_C{best_C}_siglip"
else:
    S_final_test = S_zs_test
    tag = "zeroshot_siglip"

test_query_ids, test_cat_indices = [], []
for _, row in test_queries.iterrows():
    cat_id, _, _ = matcher.predict(str(row["description"]))
    test_query_ids.append(str(row["query_id"]))
    test_cat_indices.append(cat_id2idx.get(cat_id, 0))

sub_df   = build_submission(S_final_test, test_ids, test_query_ids, test_cat_indices, top=20)
is_valid = validate_submission(sub_df, test_gallery)

ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
sub_path = OUT_DIR / "submissions" / f"{ts}_{tag}.csv"
sub_df.to_csv(sub_path, index=False)
sub_df.to_csv(OUT_DIR / "submission.csv", index=False)
print(f"Saved: {sub_path}")

# Official validator
import subprocess, sys
r = subprocess.run(
    [sys.executable, str(DATA_DIR/"validate_submission.py"),
     str(OUT_DIR/"submission.csv"), "--data-dir", str(DATA_DIR)],
    capture_output=True, text=True)
print(r.stdout)
if r.stderr: print(r.stderr)
print("Validator exit code:", r.returncode)

In [ ]:
# ── 8. Experiment log rows ────────────────────────────────────────────────────
today = datetime.date.today()
print("Add to reports/experiment_log.md:\n")
for i, r in enumerate(results[1:], start=4):  # E004 onwards
    print(f"| E{i:03d} | {today} | 4 | {r['method']} on SigLIP feats | seed=42 | 42 | "
          f"{r['r_precision']:.4f} | {r['map@20']:.4f} | {r['p@5']:.4f} | "
          f"{r['hit@1']:.4f} | {r['hit@5']:.4f} | {r['mrr@20']:.4f} | phase4 |")